# 03_limpieza_datos

Proyecto ARIMA / ARIMAX
Modelación epidemiológica con variables meteorológicas.

# Obtención de los datos epidemiológicos 

In [1]:
# Inicio del análisis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  


In [7]:
# Importar datos epidemiológicos 
ubicacion_epi_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Secretaria de salud\BD_DENGUE_2021-2025_CAUCASIA.xlsx"
ubicacion_epi_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\2_datos_version_marzo_25_2026\epidemiologicos.xlsx"
df_epi_caucasia=pd.read_excel(ubicacion_epi_janis)

# Importar datos meteorológicos 
ubicacion_meteo_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos meteorológicos\Datos_NS_2021-2025.xlsx"
ubicacion_meteo_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\2_datos_version_marzo_25_2026\meteorologicos.xlsx"
df_meteo_caucasia=pd.read_excel(ubicacion_meteo_janis)



In [8]:
df_epi_caucasia.head(2)

,ini_sin_,semana,año,area_,localidad_,cen_pobla_,vereda_,bar_ver_,dir_res_,nmun_proce,nmun_resi
0,2021-01-21,3,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CR 39 E 48 C SUR 36,CAUCASIA,ENVIGADO
1,2021-02-08,6,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CL 29 A 42-99,CAUCASIA,BOGOTA


In [9]:
df_meteo_caucasia.head(2)

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,QV2M,RH2M,PRECTOTCORR,WS2M,WS2M_MAX,WS2M_MIN,ALLSKY_SFC_UV_INDEX
0,2021,3,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
1,2021,4,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47


In [10]:
df_meteo_caucasia.columns

Index(['YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'QV2M', 'RH2M',
       'PRECTOTCORR', 'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'ALLSKY_SFC_UV_INDEX'],
      dtype='object')

In [11]:
df_epi_caucasia.columns 

Index(['ini_sin_', 'semana', 'año', 'area_', 'localidad_', 'cen_pobla_',
       'vereda_', 'bar_ver_', 'dir_res_', 'nmun_proce', 'nmun_resi'],
      dtype='object')

# Remuestreo de los datos Epidemiológicos a frecuencia semanal 

In [15]:
# remuestreo epidemiológico 2024 y 2025. 2021 al 2024 con 52 semanas, 2025 con 53 semanas
# ---------------------------
# 1. AGRUPAR CASOS POR SEMANA REAL
# ---------------------------
df_epi = df_epi_caucasia.copy()

df_semanal = df_epi.groupby(['año', 'semana']).size().reset_index(name='num_casos')


# ---------------------------
# 2. ESTRUCTURA 2021–2024 (52 semanas)
# ---------------------------
estructura_1 = pd.MultiIndex.from_product(
    [range(2021, 2025), range(1, 53)],
    names=['año', 'semana']
).to_frame(index=False)

df_1_epi = pd.merge(estructura_1, df_semanal, on=['año', 'semana'], how='left')
df_1_epi['num_casos'] = df_1_epi['num_casos'].fillna(0).astype(int)


# ---------------------------
# 3. ESTRUCTURA 2025 (53 semanas)
# ---------------------------
estructura_2 = pd.DataFrame({
    'año': [2025]*53,
    'semana': range(1, 54)
})

df_2_epi = pd.merge(estructura_2, df_semanal, on=['año', 'semana'], how='left')
df_2_epi['num_casos'] = df_2_epi['num_casos'].fillna(0).astype(int)


# ---------------------------
# 4. FECHAS CORRECTAS (CLAVE)
# ---------------------------
inicios = {
    2021: '2021-01-03',
    2022: '2022-01-02',
    2023: '2023-01-01',
    2024: '2023-12-31',  # ← corrige el error de 25 vs 22 dic
    2025: '2024-12-29'
}

def asignar_fecha(row):
    inicio = pd.Timestamp(inicios[row['año']])
    return inicio + pd.to_timedelta((row['semana'] - 1) * 7, unit='D')


# Aplicar
df_1_epi['fecha'] = df_1_epi.apply(asignar_fecha, axis=1)
df_2_epi['fecha'] = df_2_epi.apply(asignar_fecha, axis=1)


# ---------------------------
# 5. UNIR TODO
# ---------------------------
df_final_epi = pd.concat([df_1_epi, df_2_epi])

df_final_epi = df_final_epi.sort_values('fecha').set_index('fecha')
df_final_epi = df_final_epi.rename(columns={'semana': 'semana_epi'})

In [13]:
df_1_epi

,año,semana,num_casos,fecha
0,2021,1,0,2021-01-03
1,2021,2,0,2021-01-10
2,2021,3,1,2021-01-17
3,2021,4,0,2021-01-24
4,2021,5,0,2021-01-31
...,...,...,...,...
203,2024,48,68,2024-11-24
204,2024,49,92,2024-12-01
205,2024,50,85,2024-12-08
206,2024,51,74,2024-12-15


In [14]:
df_2_epi

,año,semana,num_casos,fecha
0,2025,1,92,2024-12-29
1,2025,2,90,2025-01-05
2,2025,3,82,2025-01-12
3,2025,4,99,2025-01-19
4,2025,5,103,2025-01-26
5,2025,6,106,2025-02-02
6,2025,7,71,2025-02-09
7,2025,8,78,2025-02-16
8,2025,9,52,2025-02-23
9,2025,10,83,2025-03-02


### Meteorológicos

In [16]:
# Renombrar columnas del dataframe nasa
df_meteo_caucasia.rename(columns={
    'YEAR': 'año',
    'DOY': 'dia',
    'T2M': 'temp',
    'T2M_MAX': 'temp_max',
    'T2M_MIN': 'temp_min',
    'QV2M': 'hum_esp',
    'RH2M': 'hum_rel',
    'PRECTOTCORR': 'prec',
    'WS2M': 'vel_vi',
    'WS2M_MAX': 'vel_vi_max',
    'WS2M_MIN': 'vel_vi_min',
    'ALLSKY_SFC_UV_INDEX': 'uv'
}, inplace=True)
df_meteo_caucasia.head(3)

,año,dia,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021,3,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
1,2021,4,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47
2,2021,5,27.84,34.96,22.71,15.16,68.47,0.62,0.16,0.34,0.03,2.30


In [17]:
df_meteo_caucasia.columns

Index(['año', 'dia', 'temp', 'temp_max', 'temp_min', 'hum_esp', 'hum_rel',
       'prec', 'vel_vi', 'vel_vi_max', 'vel_vi_min', 'uv'],
      dtype='object')

In [18]:
df_meteo = df_meteo_caucasia.copy()

# Crear fecha real
df_meteo['fecha'] = pd.to_datetime(df_meteo['año'].astype(str) + '-' + df_meteo['dia'].astype(str), format='%Y-%j')

In [19]:
df_meteo['dias_lluvia'] = (df_meteo['prec'] >= 1).astype(int)

In [20]:
inicios = {
    2021: pd.Timestamp('2021-01-03'),
    2022: pd.Timestamp('2022-01-02'),
    2023: pd.Timestamp('2023-01-01'),
    2024: pd.Timestamp('2023-12-31'),
    2025: pd.Timestamp('2024-12-29')
}

def asignar_semana(row):
    año = row['fecha'].year
    
    # Ajuste especial para fechas que pertenecen a semana epi del siguiente año
    if row['fecha'] >= inicios[2025]:
        año = 2025
    elif row['fecha'] >= inicios[2024]:
        año = 2024
    elif row['fecha'] >= inicios[2023]:
        año = 2023
    elif row['fecha'] >= inicios[2022]:
        año = 2022
    else:
        año = 2021

    semana = ((row['fecha'] - inicios[año]).days // 7) + 1
    return pd.Series([año, semana])

df_meteo[['año_epi', 'semana']] = df_meteo.apply(asignar_semana, axis=1)

In [21]:
df_semanal = df_meteo.groupby(['año_epi', 'semana']).agg({
    'temp': 'mean',
    'temp_max': 'mean',
    'temp_min': 'mean',
    'hum_esp': 'mean',
    'hum_rel': 'mean',
    'prec': 'mean',
    'dias_lluvia': 'sum',
    'vel_vi': 'mean',
    'vel_vi_max': 'mean',
    'vel_vi_min': 'mean',
    'uv': 'mean',
    'prec': 'sum',
}).reset_index()

df_semanal = df_semanal.rename(columns={'año_epi': 'año'})

In [22]:
estructura_1 = pd.MultiIndex.from_product(
    [range(2021, 2025), range(1, 53)],
    names=['año', 'semana']
).to_frame(index=False)

df_1_meteo = pd.merge(estructura_1, df_semanal, on=['año', 'semana'], how='left')

In [23]:
estructura_2 = pd.DataFrame({
    'año': [2025]*53,
    'semana': range(1, 54)
})

df_2_meteo = pd.merge(estructura_2, df_semanal, on=['año', 'semana'], how='left')

In [24]:
def asignar_fecha(row):
    inicio = inicios[row['año']]
    return inicio + pd.to_timedelta((row['semana'] - 1) * 7, unit='D')

df_1_meteo['fecha'] = df_1_meteo.apply(asignar_fecha, axis=1)
df_2_meteo['fecha'] = df_2_meteo.apply(asignar_fecha, axis=1)

In [25]:
df_final_meteo = pd.concat([df_1_meteo, df_2_meteo])
df_final_meteo = df_final_meteo.sort_values('fecha').set_index('fecha')

In [26]:
df_1_meteo

,año,semana,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,fecha
0,2021,1,28.252857,34.200000,23.832857,16.308571,70.508571,5.72,1,0.121429,0.244286,0.047143,2.222857,2021-01-03
1,2021,2,28.687143,34.910000,24.195714,17.318571,72.885714,19.15,5,0.117143,0.208571,0.038571,2.254286,2021-01-10
2,2021,3,29.592857,36.372857,24.090000,16.112857,65.122857,0.77,0,0.124286,0.225714,0.045714,2.420000,2021-01-17
3,2021,4,29.190000,35.978571,24.200000,16.511429,68.068571,12.92,4,0.122857,0.220000,0.035714,2.477143,2021-01-24
4,2021,5,29.454286,35.882857,24.684286,17.427143,70.105714,17.94,3,0.120000,0.221429,0.030000,2.290000,2021-01-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,2024,48,26.315714,29.455714,24.061429,19.802857,92.042857,42.79,7,0.154286,0.368571,0.040000,1.878571,2024-11-24
204,2024,49,26.521429,29.908571,24.115714,19.595714,90.112857,18.76,3,0.157143,0.357143,0.054286,1.762857,2024-12-01
205,2024,50,26.795714,30.527143,23.571429,19.075714,86.485714,7.92,1,0.194286,0.521429,0.021429,1.921429,2024-12-08
206,2024,51,26.630000,29.824286,24.062857,19.545714,89.310000,10.38,3,0.155714,0.425714,0.021429,1.652857,2024-12-15


In [27]:
df_2_meteo

,año,semana,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,fecha
0,2025,1,25.131429,28.048571,23.124286,18.568571,92.692857,72.06,7,0.160000,0.390000,0.030000,1.744286,2024-12-29
1,2025,2,26.591429,30.398571,23.624286,18.684286,86.071429,11.90,2,0.187143,0.467143,0.040000,1.871429,2025-01-05
2,2025,3,26.794286,31.061429,23.178571,18.380000,84.000000,0.71,0,0.180000,0.474286,0.032857,2.080000,2025-01-12
3,2025,4,26.942857,31.498571,23.451429,18.291429,82.930000,2.61,2,0.162857,0.415714,0.008571,1.988571,2025-01-19
4,2025,5,27.131429,31.527143,24.010000,19.011429,84.907143,18.51,1,0.120000,0.242857,0.017143,1.947143,2025-01-26
5,2025,6,27.028571,31.348571,23.721429,18.410000,83.014286,35.14,5,0.135714,0.292857,0.022857,1.775714,2025-02-02
6,2025,7,26.948571,31.570000,23.697143,18.337143,83.350000,26.70,7,0.125714,0.248571,0.024286,1.754286,2025-02-09
7,2025,8,27.531429,32.497143,24.010000,18.410000,81.182857,16.16,6,0.142857,0.280000,0.042857,1.890000,2025-02-16
8,2025,9,27.650000,32.394286,24.411429,18.812857,82.111429,25.09,5,0.134286,0.231429,0.060000,1.805714,2025-02-23
9,2025,10,27.300000,32.092857,24.110000,18.254286,81.497143,23.63,6,0.137143,0.232857,0.060000,1.942857,2025-03-02


# Unir las bases de datos epidemiológicas con la base de datos meteorológica de acuerdo al criterop año semana epidemiológica 


In [28]:
# Unir datos
df_final_fusionado = (
    pd.merge(
        df_final_epi.reset_index(),
        df_final_meteo.reset_index().drop(columns=['año', 'semana']),
        on='fecha',
        how='left'
    )
    .rename(columns={'num_casos': 'casos_dengue'})
)

# Quitar hora de la fecha
df_final_fusionado['fecha'] = pd.to_datetime(df_final_fusionado['fecha']).dt.date

# Reordenar columnas
columnas_orden = [
    'fecha',
    'año',
    'semana_epi',
    'casos_dengue',
    'temp', 'temp_max', 'temp_min',
    'hum_esp', 'hum_rel',
    'prec', 'dias_lluvia',
    'vel_vi', 'vel_vi_max', 'vel_vi_min',
    'uv'
]

df_final_fusionado = df_final_fusionado[columnas_orden]

# Guardar
df_final_fusionado.to_excel('df_final_fusionado.xlsx', index=False)

df_final_fusionado.head()

,fecha,año,semana_epi,casos_dengue,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021-01-03,2021,1,0,28.252857,34.200000,23.832857,16.308571,70.508571,5.72,1,0.121429,0.244286,0.047143,2.222857
1,2021-01-10,2021,2,0,28.687143,34.910000,24.195714,17.318571,72.885714,19.15,5,0.117143,0.208571,0.038571,2.254286
2,2021-01-17,2021,3,1,29.592857,36.372857,24.090000,16.112857,65.122857,0.77,0,0.124286,0.225714,0.045714,2.420000
3,2021-01-24,2021,4,0,29.190000,35.978571,24.200000,16.511429,68.068571,12.92,4,0.122857,0.220000,0.035714,2.477143
4,2021-01-31,2021,5,0,29.454286,35.882857,24.684286,17.427143,70.105714,17.94,3,0.120000,0.221429,0.030000,2.290000


# Convertir el DataFrame df_epi_meteo en un .excel

In [29]:
# Guardar la base de datos fusionada en un nuevo archivo Excel
df_final_fusionado.to_excel(r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Secretaria de salud\BD_FUSIONADA_DENGUE_2021-2025.xlsx", index=False)


Hasta aquí hemos logrado la estructuración de la base de datos epidemiológica y meteorológica.

Tarea de resumir esta limpieza, **remuestreo** y fusion de datos, en una sola diapositiva. 